# ABMAP training walkthrough

This notebook mirrors the command-line workflow for reproducing the ABMAP training run described in doi:10.1073/pnas.2418918121.

## 1. Environment setup

Install the project dependencies (run this once per environment).

In [ ]:
!pip install -r ../requirements.txt

## 2. Download the supplementary datasets

Run the helper script to fetch the Excel workbooks referenced in the paper. If you are working offline, manually place the files listed in `configs/abmap_full.yaml` inside `../data/raw/` before executing the next cells.

In [ ]:
%run ../scripts/download_abmap_dataset.py ../data/raw

## 3. Build the processed dataset

Convert the raw supplementary tables into the consolidated Parquet file that the trainer expects.

In [ ]:
%run ../scripts/prepare_abmap_dataset.py ../data/raw/pnas.2418918121.sd01.xlsx ../data/raw/pnas.2418918121.sd02.xlsx --output ../data/processed/abmap_training.parquet

## 4. Launch the training loop

Train the heavy/light + antigen interaction model using the full dataset hyperparameters derived from the manuscript.

In [ ]:
%run ../scripts/train_abmap.py ../configs/abmap_full.yaml --output-dir ../artifacts/notebook_run

## 5. Review metrics

Load the JSON training history emitted by the trainer and plot the learning curves.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

history_path = Path('../artifacts/notebook_run/training_history.json')
with history_path.open() as fh:
    history = json.load(fh)
df = pd.DataFrame({
    'epoch': history['epochs'],
    'train_loss': history['train_loss'],
    'val_loss': history['val_loss'],
    'spearman': [metrics.get('spearman') for metrics in history['val_metrics']],
})
df.set_index('epoch')[['train_loss', 'val_loss']].plot(figsize=(8, 4))
plt.title('Loss curves')
plt.show()
df.set_index('epoch')['spearman'].plot(figsize=(8, 4))
plt.title('Validation Spearman correlation')
plt.show()